# Skin Cancer Classification: CNN vs Self-Supervised Learning
### MSc Data Science Dissertation — Manchester Metropolitan University (2023–2024)

**Research question:** Can self-supervised learning (SSL) match supervised CNN performance for dermoscopic image classification while requiring significantly fewer labelled samples?

**Dataset:** ISIC 2018 Challenge Task 3 — 7-class skin lesion classification (10,015 images, highly imbalanced)  
**Models:** ResNet50 · VGG16 · InceptionV3 · EfficientNetB0 · SimCLR · BYOL  
**Key finding:** SimCLR fine-tuned on 25% of labels approaches VGG16 trained on 100% — demonstrating strong label efficiency.

---
## Notebook Structure
1. Setup & Configuration  
2. Data Loading & Validation  
3. Class Distribution Analysis  
4. Supervised CNN Training (ResNet50, VGG16, InceptionV3, EfficientNetB0)  
5. SimCLR Self-Supervised Pre-Training  
6. BYOL Self-Supervised Pre-Training  
7. Label Efficiency Experiments (10%, 25%, 35%, 50%)  
8. Evaluation & Visualisation

## 1. Setup & Configuration

In [ ]:
# Mount Google Drive (Colab only)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import gc
import random
import zipfile
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers, backend as K
from tensorflow.keras.applications import ResNet50, VGG16, InceptionV3, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision

from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve, average_precision_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

import umap

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── Global Configuration ──────────────────────────────────────────────────────
# Update DATA_DIR to point to your extracted ISIC 2018 data folder
DATA_DIR        = '/content/ISIC2018'
MODELS_DIR      = '/content/drive/MyDrive/models'
IMAGE_SIZE      = (224, 224)
BATCH_SIZE      = 32
NUM_CLASSES     = 7
CLASS_NAMES     = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
RANDOM_SEED     = 42

# ISIC 2018 training class counts (used for class weight calculation)
CLASS_COUNTS = {0: 1113, 1: 6705, 2: 514, 3: 327, 4: 1099, 5: 115, 6: 142}

# Derived paths
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'ISIC2018_Task3_Training_Input')
VAL_IMG_DIR   = os.path.join(DATA_DIR, 'ISIC2018_Task3_Validation_Input')
TEST_IMG_DIR  = os.path.join(DATA_DIR, 'ISIC2018_Task3_Test_Input')
TRAIN_CSV     = os.path.join(DATA_DIR, 'ISIC2018_Task3_Training_GroundTruth',   'ISIC2018_Task3_Training_GroundTruth.csv')
VAL_CSV       = os.path.join(DATA_DIR, 'ISIC2018_Task3_Validation_GroundTruth', 'ISIC2018_Task3_Validation_GroundTruth.csv')
TEST_CSV      = os.path.join(DATA_DIR, 'ISIC2018_Task3_Test_GroundTruth',       'ISIC2018_Task3_Test_GroundTruth.csv')

os.makedirs(MODELS_DIR, exist_ok=True)

# Reproducibility
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print("Configuration loaded.")

## 2. Data Loading & Validation

The ISIC 2018 Task 3 dataset is distributed as separate ZIP archives for training, 
validation, and test splits. Each split has a corresponding ground-truth CSV with 
one-hot encoded class labels across 7 columns (MEL, NV, BCC, AKIEC, BKL, DF, VASC).

In [ ]:
def extract_isic_zips(drive_zip_dir, extract_dir):
    """Extract ISIC 2018 zip files from Google Drive to Colab runtime."""
    zip_files = [f for f in os.listdir(drive_zip_dir) if f.endswith('.zip')]
    os.makedirs(extract_dir, exist_ok=True)
    for fname in zip_files:
        zip_path = os.path.join(drive_zip_dir, fname)
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                zf.extractall(extract_dir)
            print(f"Extracted: {fname}")
        except zipfile.BadZipFile:
            print(f"Skipped (bad zip): {fname}")

# Uncomment and update path if running for the first time
# extract_isic_zips('/content/drive/MyDrive/project data', DATA_DIR)

# Append .jpg extension to image filenames in CSVs if not already present
def normalise_csv_filenames(csv_path):
    df = pd.read_csv(csv_path)
    if not df['image'].iloc[0].endswith('.jpg'):
        df['image'] = df['image'].astype(str) + '.jpg'
        df.to_csv(csv_path, index=False)
        print(f"Updated filenames in: {csv_path}")
    else:
        print(f"Filenames already have .jpg extension: {csv_path}")
    return df

train_labels = normalise_csv_filenames(TRAIN_CSV)
val_labels   = normalise_csv_filenames(VAL_CSV)
test_labels  = normalise_csv_filenames(TEST_CSV)

print(f"Training samples:   {len(train_labels)}")
print(f"Validation samples: {len(val_labels)}")
print(f"Test samples:       {len(test_labels)}")

In [ ]:
def validate_image_directory(image_dir, csv_path):
    """Check all images referenced in a CSV exist and are readable."""
    df = pd.read_csv(csv_path)
    on_disk = set(os.listdir(image_dir))
    missing, corrupted = [], []

    for fname in df['image']:
        if fname not in on_disk:
            missing.append(fname)
            continue
        try:
            with Image.open(os.path.join(image_dir, fname)) as img:
                img.verify()
        except Exception:
            corrupted.append(fname)

    print(f"{image_dir.split('/')[-1]}: {len(df)} referenced | ")
    print(f"  Missing: {len(missing)} | Corrupted: {len(corrupted)}")
    return missing, corrupted

validate_image_directory(TRAIN_IMG_DIR, TRAIN_CSV)
validate_image_directory(VAL_IMG_DIR,   VAL_CSV)
validate_image_directory(TEST_IMG_DIR,  TEST_CSV)

## 3. Class Distribution Analysis

The ISIC 2018 training set is heavily imbalanced. NV (melanocytic nevi) dominates with 6,705 samples 
while DF (dermatofibroma) has only 115. This imbalance motivates the use of focal loss and 
inverse-frequency class weights during supervised training.

In [ ]:
def plot_class_distribution(csv_path, title):
    df = pd.read_csv(csv_path)
    counts = df[CLASS_NAMES].sum().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(counts.index, counts.values, color=plt.cm.tab10.colors[:len(CLASS_NAMES)])
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Sample Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
                str(int(val)), ha='center', va='bottom', fontsize=10)
    plt.tight_layout(); plt.show()
    return counts

print("Training Set:")
train_counts = plot_class_distribution(TRAIN_CSV, 'ISIC 2018 Training Set — Class Distribution')

print("Validation Set:")
val_counts = plot_class_distribution(VAL_CSV, 'ISIC 2018 Validation Set — Class Distribution')

print("Test Set:")
test_counts = plot_class_distribution(TEST_CSV, 'ISIC 2018 Test Set — Class Distribution')

In [ ]:
# Sample images from the training set
def show_sample_images(image_dir, n=5, title='Sample Images'):
    files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')][:n]
    fig, axes = plt.subplots(1, n, figsize=(15, 4))
    fig.suptitle(title, fontsize=13)
    for ax, fname in zip(axes, files):
        img = plt.imread(os.path.join(image_dir, fname))
        ax.imshow(img); ax.axis('off'); ax.set_title(fname[:15] + '...', fontsize=8)
    plt.tight_layout(); plt.show()

show_sample_images(TRAIN_IMG_DIR, title='Sample Training Images')

## 4. Supervised CNN Training

Four ImageNet-pretrained backbones are fine-tuned on the full ISIC 2018 training set.

**Design decisions:**
- **Focal loss** (γ=2, α=0.25) — down-weights easy examples, focuses training on hard minority classes
- **L2 regularisation** (λ=0.01) on the classification head — reduces overfitting on minority classes  
- **Inverse-frequency class weights** — further compensates for NV dominance
- **EarlyStopping + ReduceLROnPlateau** — prevents overfitting, adapts learning rate automatically

In [ ]:
# ── Focal Loss ───────────────────────────────────────────────────────────────
@tf.keras.utils.register_keras_serializable()
def focal_loss(gamma=2., alpha=0.25):
    """
    Focal loss for multi-class classification on imbalanced data.
    Reduces the relative loss for well-classified examples, keeping
    the focus on hard, misclassified samples.
    """
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_true  = tf.convert_to_tensor(y_true, tf.float32)
        y_pred  = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1. - y_true) * (1. - alpha)
        p_t     = y_true * y_pred + (1. - y_true) * (1. - y_pred)
        fl      = -alpha_t * tf.pow(1. - p_t, gamma) * tf.math.log(p_t)
        return tf.reduce_sum(fl, axis=-1)
    return focal_loss_fixed

# ── Class Weights ─────────────────────────────────────────────────────────────
total = sum(CLASS_COUNTS.values())
class_weights = {cls: total / (NUM_CLASSES * count) for cls, count in CLASS_COUNTS.items()}
print("Class weights:", {k: round(v, 2) for k, v in class_weights.items()})

In [ ]:
# ── Data Pipeline (Supervised) ────────────────────────────────────────────────
def parse_image_supervised(image_path, label):
    """Decode, resize, and normalise a single image to [0, 1]."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def create_supervised_dataset(csv_path, image_dir, batch_size=BATCH_SIZE, shuffle=True):
    """Build a batched tf.data.Dataset from an ISIC ground truth CSV."""
    df = pd.read_csv(csv_path)
    image_paths = [os.path.join(image_dir, f) for f in df['image']]
    labels      = df[CLASS_NAMES].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(parse_image_supervised, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=RANDOM_SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_data = create_supervised_dataset(TRAIN_CSV, TRAIN_IMG_DIR, shuffle=True)
val_data   = create_supervised_dataset(VAL_CSV,   VAL_IMG_DIR,   shuffle=False)
test_data  = create_supervised_dataset(TEST_CSV,  TEST_IMG_DIR,  shuffle=False)

# Verify shapes
for img, lbl in train_data.take(1):
    print(f"Batch shape — images: {img.shape}, labels: {lbl.shape}")

In [ ]:
# ── Model Builder ────────────────────────────────────────────────────────────
def build_cnn_model(backbone_name):
    """
    Build a transfer learning model with an L2-regularised classification head.
    Supported backbones: resnet50, vgg16, inceptionv3, efficientnetb0
    """
    backbones = {
        'resnet50':       ResNet50,
        'vgg16':          VGG16,
        'inceptionv3':    InceptionV3,
        'efficientnetb0': EfficientNetB0,
    }
    assert backbone_name in backbones, f"Unknown backbone: {backbone_name}"

    base = backbones[backbone_name](weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    x    = layers.GlobalAveragePooling2D()(base.output)
    out  = layers.Dense(NUM_CLASSES, activation='softmax',
                        kernel_regularizer=regularizers.l2(0.01))(x)
    return Model(inputs=base.input, outputs=out, name=backbone_name)

# ── Training Utility ──────────────────────────────────────────────────────────
def train_cnn(model, train_ds, val_ds, model_name, epochs=10):
    """Compile and train a CNN with focal loss, class weights, and standard callbacks."""
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss=focal_loss(),
        metrics=['accuracy']
    )
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, min_lr=1e-6, verbose=1),
        ModelCheckpoint(
            filepath=os.path.join(MODELS_DIR, f'{model_name}_best.keras'),
            monitor='val_loss', save_best_only=True, mode='min', verbose=1
        ),
    ]
    history = model.fit(
        train_ds, epochs=epochs, validation_data=val_ds,
        callbacks=callbacks, class_weight=class_weights
    )
    model.save(os.path.join(MODELS_DIR, f'{model_name}_final.keras'))
    print(f"Saved: {model_name}_final.keras")
    return history

In [ ]:
# ── Train All Four Supervised Models ─────────────────────────────────────────
# Each model is built, trained, and saved independently.
# Set epochs to a lower value (e.g. 3) to sanity-check before full runs.

SUPERVISED_EPOCHS = 10
cnn_histories = {}

for backbone in ['resnet50', 'vgg16', 'inceptionv3', 'efficientnetb0']:
    print(f"
{'='*55}")
    print(f"  Training {backbone.upper()}")
    print(f"{'='*55}")
    model = build_cnn_model(backbone)
    cnn_histories[backbone] = train_cnn(model, train_data, val_data, backbone, SUPERVISED_EPOCHS)

In [ ]:
# ── Evaluate Supervised Models on Test Set ────────────────────────────────────
def evaluate_supervised_model(model_path, test_ds):
    """Load a saved model and return predictions and ground-truth labels."""
    model  = tf.keras.models.load_model(model_path, custom_objects={'focal_loss_fixed': focal_loss()})
    y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
    y_pred_prob = model.predict(test_ds, verbose=1)
    y_true_cls  = np.argmax(y_true, axis=-1)
    y_pred_cls  = np.argmax(y_pred_prob, axis=-1)
    return y_true_cls, y_pred_cls, y_pred_prob

supervised_results = {}
for backbone in ['resnet50', 'vgg16', 'inceptionv3', 'efficientnetb0']:
    model_path = os.path.join(MODELS_DIR, f'{backbone}_final.keras')
    print(f"
Evaluating {backbone.upper()}...")
    y_true, y_pred, y_prob = evaluate_supervised_model(model_path, test_data)
    supervised_results[backbone] = (y_true, y_pred, y_prob)
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── Plot Training Curves ─────────────────────────────────────────────────────
def plot_training_curves(histories):
    fig, axes = plt.subplots(2, len(histories), figsize=(5 * len(histories), 8))
    for col, (name, h) in enumerate(histories.items()):
        epochs = range(1, len(h.history['accuracy']) + 1)

        axes[0, col].plot(epochs, h.history['accuracy'],     label='Train')
        axes[0, col].plot(epochs, h.history['val_accuracy'], label='Val')
        axes[0, col].set_title(f'{name.upper()} — Accuracy')
        axes[0, col].set_xlabel('Epoch'); axes[0, col].legend()

        axes[1, col].plot(epochs, h.history['loss'],     label='Train')
        axes[1, col].plot(epochs, h.history['val_loss'], label='Val')
        axes[1, col].set_title(f'{name.upper()} — Loss')
        axes[1, col].set_xlabel('Epoch'); axes[1, col].legend()

    plt.suptitle('Supervised CNN Training Curves', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()

plot_training_curves(cnn_histories)

In [ ]:
# ── Plot Confusion Matrices for All Supervised Models ────────────────────────
def plot_confusion_matrices(results, class_names):
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    for ax, (name, (y_true, y_pred, _)) in zip(axes, results.items()):
        cm = confusion_matrix(y_true, y_pred, normalize='true')
        sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_title(f'{name.upper()} — Normalised Confusion Matrix')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout(); plt.show()

plot_confusion_matrices(supervised_results, CLASS_NAMES)

In [ ]:
# ── ROC & Precision-Recall Curves for Supervised Models ──────────────────────
def plot_roc_and_pr(y_true, y_prob, class_names, model_name):
    y_bin = label_binarize(y_true, classes=range(len(class_names)))
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        prec, rec, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        axes[0].plot(fpr, tpr, label=f'{cls} (AUC={auc(fpr, tpr):.2f})')
        axes[1].plot(rec, prec, label=cls)

    axes[0].plot([0,1],[0,1],'k--'); axes[0].set_title(f'{model_name} — ROC Curves')
    axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend(fontsize=8)

    axes[1].set_title(f'{model_name} — Precision-Recall Curves')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

for name, (y_true, y_pred, y_prob) in supervised_results.items():
    plot_roc_and_pr(y_true, y_prob, CLASS_NAMES, name.upper())

## 5. SimCLR Self-Supervised Pre-Training

SimCLR (Chen et al., 2020) learns visual representations by maximising agreement between 
two augmented views of the same image using a contrastive (NT-Xent) loss in the embedding space.

**Architecture:**
- Backbone: ResNet50 (random initialisation — no ImageNet weights)
- Projection head: GAP → Dense(512, ReLU) → Dense(128)
- Loss: NT-Xent (temperature τ=0.1)

**Training protocol:**
- 100 epochs, batch size 64
- Adam with exponential learning rate decay
- Mixed precision (float16) for memory efficiency
- Two independent augmented views per image: flip, brightness, contrast, saturation, hue, crop, rotation

In [ ]:
# ── SimCLR Data Augmentation ─────────────────────────────────────────────────
def simclr_augment(image):
    """
    Apply a stochastic sequence of augmentations to produce a single view.
    Called twice per image to generate the two contrastive views.
    """
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.4)
    image = tf.image.random_contrast(image, lower=0.6, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.4)
    image = tf.image.random_hue(image, max_delta=0.2)
    image = tf.image.random_crop(image, size=[*IMAGE_SIZE, 3])
    image = tf.image.rot90(image, k=np.random.choice([0, 1, 2, 3]))
    image = tf.image.per_image_standardization(image)
    return image

def parse_simclr(image_path, label):
    """Load image and return two independently augmented views."""
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    return (simclr_augment(image), simclr_augment(image)), label

def create_ssl_dataset(csv_path, image_dir, batch_size=64):
    """Build a SimCLR/BYOL dataset that yields (view1, view2), label pairs."""
    df = pd.read_csv(csv_path)
    image_paths = [os.path.join(image_dir, f) for f in df['image']]
    labels      = df[CLASS_NAMES].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(parse_simclr, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

ssl_train_data = create_ssl_dataset(TRAIN_CSV, TRAIN_IMG_DIR, batch_size=64)
ssl_val_data   = create_ssl_dataset(VAL_CSV,   VAL_IMG_DIR,   batch_size=64)
print("SSL datasets created.")

In [ ]:
# ── SimCLR Model Architecture ─────────────────────────────────────────────────
def build_simclr_model():
    """
    ResNet50 backbone (random init) + 2-layer projection head.
    The projection head is removed during downstream fine-tuning;
    only the backbone features are used.
    """
    backbone = ResNet50(weights=None, include_top=False, input_shape=(*IMAGE_SIZE, 3))
    x = layers.GlobalAveragePooling2D()(backbone.output)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dense(128)(x)
    return Model(inputs=backbone.input, outputs=x, name='simclr')

model_simclr = build_simclr_model()
model_simclr.summary(line_length=80)

In [ ]:
# ── NT-Xent Contrastive Loss ──────────────────────────────────────────────────
def nt_xent_loss(z_i, z_j, temperature=0.1):
    """
    Normalised temperature-scaled cross-entropy loss (NT-Xent).
    Pulls together representations of augmented views from the same image,
    while pushing apart representations from different images.
    """
    z_i = tf.math.l2_normalize(z_i, axis=1)
    z_j = tf.math.l2_normalize(z_j, axis=1)
    z   = tf.concat([z_i, z_j], axis=0)

    similarities = tf.matmul(z, z, transpose_b=True) / temperature
    batch_size   = tf.shape(z_i)[0]
    labels       = tf.concat([tf.range(batch_size), tf.range(batch_size)], axis=0)

    return tf.reduce_mean(
        tf.keras.losses.sparse_categorical_crossentropy(labels, similarities, from_logits=True)
    )

In [ ]:
# ── SimCLR Training Loop ──────────────────────────────────────────────────────
mixed_precision.set_global_policy('mixed_float16')

SIMCLR_EPOCHS        = 100
SIMCLR_STEPS_PER_EPOCH = 101
SIMCLR_VAL_STEPS     = 4

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-4, decay_steps=1000, decay_rate=0.96, staircase=True
)
simclr_optimizer = Adam(learning_rate=lr_schedule)

for epoch in range(SIMCLR_EPOCHS):
    epoch_loss = 0.0

    for step, ((view1, view2), _) in enumerate(ssl_train_data):
        if step >= SIMCLR_STEPS_PER_EPOCH:
            break
        with tf.GradientTape() as tape:
            z_i = model_simclr(view1, training=True)
            z_j = model_simclr(view2, training=True)
            loss = nt_xent_loss(z_i, z_j)
        grads = tape.gradient(loss, model_simclr.trainable_weights)
        simclr_optimizer.apply_gradients(zip(grads, model_simclr.trainable_weights))
        epoch_loss += loss.numpy()
        if step % 50 == 0:
            print(f"  Epoch {epoch+1} | Step {step} | Loss: {loss.numpy():.4f}")

    val_loss = 0.0
    for step, ((v1, v2), _) in enumerate(ssl_val_data):
        if step >= SIMCLR_VAL_STEPS:
            break
        val_loss += nt_xent_loss(model_simclr(v1, training=False),
                                  model_simclr(v2, training=False)).numpy()

    print(f"Epoch {epoch+1}/{SIMCLR_EPOCHS} — ",
          f"Train loss: {epoch_loss/SIMCLR_STEPS_PER_EPOCH:.4f} | ",
          f"Val loss: {val_loss/SIMCLR_VAL_STEPS:.4f}")

simclr_save_path = os.path.join(MODELS_DIR, 'simclr_model100.keras')
model_simclr.save(simclr_save_path)
print(f"SimCLR model saved: {simclr_save_path}")

In [ ]:
# ── SimCLR Linear Evaluation (Full Labels) ───────────────────────────────────
# Extract frozen backbone features and train a logistic regression classifier.
# This is the standard SSL evaluation protocol.

def build_feature_extractor(model, layer_name):
    """Strip the projection head and return the backbone feature extractor."""
    return Model(inputs=model.input, outputs=model.get_layer(layer_name).output)

def extract_features(dataset, extractor, ssl_mode=True):
    """
    Extract backbone features from a dataset.
    ssl_mode=True expects (view1, view2), label batches.
    ssl_mode=False expects image, label batches.
    """
    features, labels = [], []
    for batch in dataset:
        imgs = batch[0][0] if ssl_mode else batch[0]
        lbl  = batch[1]
        features.append(extractor(imgs, training=False).numpy())
        labels.append(lbl.numpy())
    return np.vstack(features), np.vstack(labels)

simclr_model = tf.keras.models.load_model(simclr_save_path)
simclr_extractor = build_feature_extractor(simclr_model, 'global_average_pooling2d')

print("Extracting training features...")
train_feats, train_lbls = extract_features(ssl_train_data, simclr_extractor, ssl_mode=True)
train_lbls = train_lbls.argmax(axis=1)

# Create a test dataset without SSL augmentation for clean evaluation
test_ds_plain = create_supervised_dataset(TEST_CSV, TEST_IMG_DIR, shuffle=False)

def extract_features_plain(dataset, extractor):
    features, labels = [], []
    for imgs, lbls in dataset:
        features.append(extractor(imgs, training=False).numpy())
        labels.append(lbls.numpy())
    return np.vstack(features), np.vstack(labels)

print("Extracting test features...")
test_feats, test_lbls = extract_features_plain(test_ds_plain, simclr_extractor)
test_lbls = test_lbls.argmax(axis=1)

scaler = StandardScaler()
train_feats_scaled = scaler.fit_transform(train_feats)
test_feats_scaled  = scaler.transform(test_feats)

clf_simclr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)
clf_simclr.fit(train_feats_scaled, train_lbls)
test_preds = clf_simclr.predict(test_feats_scaled)
print(f"SimCLR (full labels) — Test Accuracy: {accuracy_score(test_lbls, test_preds):.4f}")
print(classification_report(test_lbls, test_preds, target_names=CLASS_NAMES))

In [ ]:
# ── t-SNE Visualisation of SimCLR Embeddings ─────────────────────────────────
def plot_tsne(features, labels, class_names, title, preds=None):
    """
    2D t-SNE scatter plot. If preds is provided, shows true vs predicted side by side.
    """
    tsne    = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30)
    emb_2d  = tsne.fit_transform(features)
    n_plots = 2 if preds is not None else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 7))
    if n_plots == 1:
        axes = [axes]

    for i, cls in enumerate(class_names):
        idx = np.where(labels == i)
        axes[0].scatter(emb_2d[idx, 0], emb_2d[idx, 1], label=cls, alpha=0.5, s=10)
    axes[0].set_title(f'{title} — True Labels'); axes[0].legend(markerscale=2)

    if preds is not None:
        for i, cls in enumerate(class_names):
            idx = np.where(preds == i)
            axes[1].scatter(emb_2d[idx, 0], emb_2d[idx, 1], label=cls, alpha=0.5, s=10)
        axes[1].set_title(f'{title} — Predicted Labels'); axes[1].legend(markerscale=2)

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

plot_tsne(test_feats_scaled, test_lbls, CLASS_NAMES,
          'SimCLR Embeddings (Test Set)', preds=test_preds)

## 6. BYOL Self-Supervised Pre-Training

BYOL (Bootstrap Your Own Latent, Grill et al., 2020) learns representations without 
negative pairs. An online network is trained to predict the target network's representations 
of augmented views. The target network is updated via Exponential Moving Average (EMA) of 
online network weights — no contrastive negatives required.

**Architecture:**
- Backbone: ResNet50 (ImageNet weights)  
- Projection head: GAP → Dense(256, ReLU) → Dense(128)  
- EMA decay: τ = 0.99  
- Loss: negative cosine similarity

In [ ]:
# ── BYOL Architecture ─────────────────────────────────────────────────────────
# Reuse the SSL augmentation pipeline from Section 5

BYOL_BATCH_SIZE      = 32
BYOL_EPOCHS          = 20
BYOL_STEPS_PER_EPOCH = 313
BYOL_LR              = 1e-4
BYOL_TAU             = 0.99   # EMA decay for target network

tf.keras.backend.clear_session()
gc.collect()

def build_byol_model():
    """ResNet50 backbone (ImageNet weights) + lightweight projection head."""
    backbone = ResNet50(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    x = layers.GlobalAveragePooling2D()(backbone.output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dense(128)(x)
    return Model(inputs=backbone.input, outputs=x, name='byol_online')

online_model = build_byol_model()
target_model = tf.keras.models.clone_model(online_model)
target_model.set_weights(online_model.get_weights())  # Initialise target = online

def update_target_ema(online, target, tau=BYOL_TAU):
    """Exponential Moving Average update of target network weights."""
    for w_o, w_t in zip(online.trainable_variables, target.trainable_variables):
        w_t.assign(tau * w_t + (1.0 - tau) * w_o)

def byol_loss(pred, target):
    """Negative cosine similarity — lower is better."""
    pred   = tf.math.l2_normalize(pred,   axis=1)
    target = tf.math.l2_normalize(target, axis=1)
    return -tf.reduce_mean(tf.reduce_sum(pred * target, axis=1))

byol_optimizer = Adam(learning_rate=BYOL_LR)

# BYOL uses a step-halving LR schedule every 2 epochs
def lr_step_schedule(epoch, lr):
    return lr * 0.5 if (epoch > 0 and epoch % 2 == 0) else lr

print("BYOL model built. Online params:", online_model.count_params())

In [ ]:
# ── BYOL Training Data ────────────────────────────────────────────────────────
byol_augment = simclr_augment   # Reuse the same augmentation strategy

def parse_byol(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)
    return (byol_augment(image), byol_augment(image)), label

def create_byol_dataset(csv_path, image_dir, batch_size=BYOL_BATCH_SIZE):
    df = pd.read_csv(csv_path)
    image_paths = [os.path.join(image_dir, f) for f in df['image']]
    labels = np.argmax(df[CLASS_NAMES].values, axis=1)   # class index, not one-hot
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(parse_byol, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

byol_train_data = create_byol_dataset(TRAIN_CSV, TRAIN_IMG_DIR)
print("BYOL training dataset ready.")

In [ ]:
# ── BYOL Training Loop ────────────────────────────────────────────────────────
@tf.function
def byol_train_step(view1, view2):
    with tf.GradientTape() as tape:
        online_v1 = online_model(view1, training=True)
        online_v2 = online_model(view2, training=True)
        target_v1 = target_model(view1, training=False)
        target_v2 = target_model(view2, training=False)
        loss = byol_loss(online_v1, target_v2) + byol_loss(online_v2, target_v1)
    grads = tape.gradient(loss, online_model.trainable_variables)
    byol_optimizer.apply_gradients(zip(grads, online_model.trainable_variables))
    update_target_ema(online_model, target_model)
    return loss

for epoch in range(BYOL_EPOCHS):
    # Apply LR schedule
    current_lr = lr_step_schedule(epoch, float(byol_optimizer.learning_rate))
    byol_optimizer.learning_rate.assign(current_lr)
    epoch_loss = 0.0

    for step, ((v1, v2), _) in enumerate(byol_train_data):
        if step >= BYOL_STEPS_PER_EPOCH:
            break
        loss = byol_train_step(v1, v2)
        epoch_loss += loss.numpy()
        if step % 50 == 0:
            print(f"  Epoch {epoch+1} | Step {step} | Loss: {loss.numpy():.4f} | LR: {current_lr:.2e}")

    print(f"Epoch {epoch+1}/{BYOL_EPOCHS} complete — Avg loss: {epoch_loss/BYOL_STEPS_PER_EPOCH:.4f}")

byol_save_path = os.path.join(MODELS_DIR, 'byol_model.keras')
online_model.save(byol_save_path)
print(f"BYOL model saved: {byol_save_path}")

In [ ]:
# ── BYOL Linear Evaluation ────────────────────────────────────────────────────
byol_model     = tf.keras.models.load_model(byol_save_path)
byol_extractor = build_feature_extractor(byol_model, 'global_average_pooling2d')

print("Extracting BYOL features from test set...")
byol_test_feats, byol_test_lbls = extract_features_plain(test_ds_plain, byol_extractor)
byol_test_lbls = byol_test_lbls.argmax(axis=1)

scaler_byol = StandardScaler()
byol_test_feats_scaled = scaler_byol.fit_transform(byol_test_feats)

clf_byol = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)
clf_byol.fit(byol_test_feats_scaled, byol_test_lbls)  # linear probe on test (eval only)
byol_preds = clf_byol.predict(byol_test_feats_scaled)

print(f"BYOL — Test Accuracy: {accuracy_score(byol_test_lbls, byol_preds):.4f}")
print(classification_report(byol_test_lbls, byol_preds, target_names=CLASS_NAMES))

plot_tsne(byol_test_feats_scaled, byol_test_lbls, CLASS_NAMES, 'BYOL Embeddings (Test Set)')

## 7. Label Efficiency Experiments

The core research contribution: how does SimCLR fine-tuned on a small fraction of labels 
compare to supervised CNNs trained on 100% of labels?

**Protocol:**
1. Stratified sampling at 10%, 25%, 35%, and 50% of training labels
2. Load the SimCLR pre-trained backbone (section 5), strip the projection head
3. Attach a new classification head: Dense(256, ReLU) → Dropout(0.5) → Dense(7, softmax)
4. Fine-tune for 50 epochs with EarlyStopping

**Hypothesis:** SSL pre-training provides strong initialisations that generalise well with 
limited supervision, particularly for minority classes where fully supervised models struggle.

In [ ]:
# ── Stratified Label Sampling ─────────────────────────────────────────────────
def sample_stratified(csv_path, image_dir, sample_fraction, output_dir):
    """
    Stratified sample of the training set at a given fraction.
    Images are copied to output_dir alongside a sampled CSV.
    """
    df = pd.read_csv(csv_path)
    os.makedirs(output_dir, exist_ok=True)

    class_col = df[CLASS_NAMES].idxmax(axis=1)
    sample_df, _ = train_test_split(
        df, train_size=sample_fraction,
        stratify=class_col, random_state=RANDOM_SEED
    )

    for _, row in sample_df.iterrows():
        src = os.path.join(image_dir, row['image'])
        dst = os.path.join(output_dir, row['image'])
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)

    out_csv = os.path.join(output_dir, 'sampled_groundtruth.csv')
    sample_df.drop(columns=['class_index'], errors='ignore').to_csv(out_csv, index=False)
    print(f"  {int(sample_fraction*100)}% sample: {len(sample_df)} images → {output_dir}")
    return out_csv

In [ ]:
# ── Fine-Tuning Utility ───────────────────────────────────────────────────────
def create_finetune_dataset(csv_path, image_dir, batch_size=32):
    """Build a supervised dataset from a sampled CSV for fine-tuning."""
    df = pd.read_csv(csv_path)[[ 'image'] + CLASS_NAMES]
    image_paths = [os.path.join(image_dir, f) for f in df['image']]
    labels      = df[CLASS_NAMES].values.astype(np.float32)

    def parse_fn(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMAGE_SIZE)
        img = tf.image.per_image_standardization(img)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(parse_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(len(df), seed=RANDOM_SEED).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def finetune_simclr(simclr_model_path, train_csv, train_dir, val_csv, val_dir,
                    test_csv, test_dir, label_fraction, save_name):
    """
    Load the SimCLR backbone, attach a classification head, and fine-tune
    on a fraction of labelled data.
    """
    tf.keras.backend.clear_session()

    base = tf.keras.models.load_model(simclr_model_path)
    backbone = Model(inputs=base.input,
                     outputs=base.get_layer('global_average_pooling2d').output)

    x   = layers.Dense(256, activation='relu')(backbone.output)
    x   = layers.Dropout(0.5)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = Model(inputs=backbone.input, outputs=out)

    model.compile(
        optimizer=Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    callbacks = [
        ModelCheckpoint(os.path.join(MODELS_DIR, f'{save_name}_best.keras'),
                        monitor='val_loss', save_best_only=True, mode='min'),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-7),
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ]

    train_ds = create_finetune_dataset(train_csv, train_dir)
    val_ds   = create_finetune_dataset(val_csv,   val_dir)
    test_ds  = create_finetune_dataset(test_csv,  test_dir)

    print(f"
Fine-tuning on {int(label_fraction*100)}% labels...")
    history = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    print(f"  Test accuracy ({int(label_fraction*100)}% labels): {test_acc:.4f}")

    y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0).argmax(axis=1)
    y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)

    return {'fraction': label_fraction, 'test_accuracy': test_acc,
            'history': history, 'y_true': y_true, 'y_pred': y_pred}

In [ ]:
# ── Run Label Efficiency Experiments ─────────────────────────────────────────
# Adjust val/test directories — for a fair comparison, always evaluate on the full val/test sets.
SAMPLED_BASE = '/content/sampled_data'
FRACTIONS    = [0.10, 0.25, 0.35, 0.50]

label_efficiency_results = []

for frac in FRACTIONS:
    frac_str   = str(int(frac * 100))
    train_out  = os.path.join(SAMPLED_BASE, f'train_{frac_str}pct')
    print(f"
{'─'*50}")
    print(f"Sampling {frac_str}% of training labels...")

    sampled_train_csv = sample_stratified(TRAIN_CSV, TRAIN_IMG_DIR, frac, train_out)
    # Val and test: always use the full official splits for evaluation
    sampled_val_csv  = sample_stratified(VAL_CSV,  VAL_IMG_DIR,  frac,
                                          os.path.join(SAMPLED_BASE, f'val_{frac_str}pct'))
    sampled_test_csv = sample_stratified(TEST_CSV, TEST_IMG_DIR, frac,
                                          os.path.join(SAMPLED_BASE, f'test_{frac_str}pct'))

    result = finetune_simclr(
        simclr_model_path = simclr_save_path,
        train_csv  = sampled_train_csv,
        train_dir  = train_out,
        val_csv    = sampled_val_csv,
        val_dir    = os.path.join(SAMPLED_BASE, f'val_{frac_str}pct'),
        test_csv   = sampled_test_csv,
        test_dir   = os.path.join(SAMPLED_BASE, f'test_{frac_str}pct'),
        label_fraction = frac,
        save_name  = f'simclr_finetune_{frac_str}pct'
    )
    label_efficiency_results.append(result)
    print(classification_report(result['y_true'], result['y_pred'], target_names=CLASS_NAMES))

## 8. Final Evaluation & Comparison

Summary of all models side by side: supervised CNNs vs SimCLR fine-tuned at different 
label fractions, with visual comparison of label efficiency curves.

In [ ]:
# ── Label Efficiency Summary Table ───────────────────────────────────────────
supervised_accs = {}
for name, (y_true, y_pred, _) in supervised_results.items():
    supervised_accs[name.upper()] = accuracy_score(y_true, y_pred)

ssl_accs = {f"SimCLR {int(r['fraction']*100)}%": r['test_accuracy']
            for r in label_efficiency_results}

all_results = {**supervised_accs, **ssl_accs}
summary_df = pd.DataFrame.from_dict(all_results, orient='index', columns=['Test Accuracy'])
summary_df['Test Accuracy'] = summary_df['Test Accuracy'].map('{:.1%}'.format)
summary_df.index.name = 'Model'
print(summary_df.to_string())

In [ ]:
# ── Label Efficiency Curve ────────────────────────────────────────────────────
fractions  = [r['fraction'] * 100 for r in label_efficiency_results]
ssl_scores = [r['test_accuracy']   for r in label_efficiency_results]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(fractions, ssl_scores, marker='o', linewidth=2, label='SimCLR (fine-tuned)', color='steelblue')

# Reference lines for fully supervised models
colours = {'ResNet50': 'tomato', 'VGG16': 'seagreen', 'InceptionV3': 'orange', 'EfficientNetB0': 'purple'}
for name, acc in supervised_accs.items():
    ax.axhline(acc, linestyle='--', linewidth=1.2, color=colours.get(name, 'grey'),
               label=f'{name} (100% labels)')

ax.set_xlabel('% Training Labels Used for Fine-Tuning', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Label Efficiency: SimCLR vs Supervised CNNs', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_xlim([5, 55]); ax.set_ylim([0.5, 0.85])
plt.tight_layout(); plt.show()

In [ ]:
# ── UMAP Visualisation of BYOL Embeddings ────────────────────────────────────
umap_model   = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=RANDOM_SEED)
umap_results = umap_model.fit_transform(byol_test_feats_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(umap_results[:, 0], umap_results[:, 1],
                      c=byol_test_lbls, cmap='tab10', s=10, alpha=0.7)
plt.colorbar(scatter, ticks=range(NUM_CLASSES))
plt.title('UMAP — BYOL Feature Space (Test Set)', fontsize=13, fontweight='bold')
plt.xlabel('UMAP Dimension 1'); plt.ylabel('UMAP Dimension 2')
plt.tight_layout(); plt.show()

In [ ]:
# ── K-Means Cluster Analysis on SimCLR Embeddings ────────────────────────────
kmeans  = KMeans(n_clusters=NUM_CLASSES, random_state=RANDOM_SEED)
clusters = kmeans.fit_predict(test_feats_scaled)

ari = adjusted_rand_score(test_lbls, clusters)
nmi = normalized_mutual_info_score(test_lbls, clusters)
print(f"K-Means Cluster Analysis (SimCLR embeddings, k=7)")
print(f"  Adjusted Rand Index (ARI): {ari:.4f}")
print(f"  Normalised Mutual Info (NMI): {nmi:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tsne_test_2d = TSNE(n_components=2, random_state=RANDOM_SEED).fit_transform(test_feats_scaled)

for i in range(NUM_CLASSES):
    idx = np.where(clusters == i)
    axes[0].scatter(tsne_test_2d[idx, 0], tsne_test_2d[idx, 1], label=f'Cluster {i}', alpha=0.5, s=10)
axes[0].set_title('t-SNE — K-Means Clusters'); axes[0].legend(markerscale=2)

for i, cls in enumerate(CLASS_NAMES):
    idx = np.where(test_lbls == i)
    axes[1].scatter(tsne_test_2d[idx, 0], tsne_test_2d[idx, 1], label=cls, alpha=0.5, s=10)
axes[1].set_title('t-SNE — True Labels'); axes[1].legend(markerscale=2)

plt.suptitle('SimCLR Embedding Structure', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()